# Phase 3b -- K-stress addendum (A100-40GB, High-RAM **OFF**)

Phase 2's K marginal was ~flat (0.94-1.03x) -- a real null for its regime: KV demand
peaked at ~18% of the pool, so K's capacity channel never engaged. This notebook creates
the pressure deliberately and runs on the **40GB** A100 (High-RAM OFF -- the toggle pins
the variant, PREREQ_RESULTS Check 1), because on 40GB **both** ceilings fit inside a
small grid:

- pool ~ 0.85x40GB - ~16GB weights - overhead ~ **~120-135k FP16-KV tokens**
- per-request context ~ 7.4k-token unique document + 256 output ~ 7.7k tokens
- -> **FP16-KV ceiling ~ 16 concurrent requests; FP8-KV ~ 32**

Base grid: {FP16-KV, FP8-KV} x concurrency **{8, 16, 32, 48}** x 2 repeats = 16 cells,
2 server launches. Each level has a job: 8 = below both ceilings (the Phase-2 null must
reproduce), 16 = ON the FP16 knee, 32 = FP16 saturated / ON the FP8 knee (max
divergence), 48 = beyond both -- FP8's own plateau at ~2x FP16's, the demonstration the
80GB design could never fit in a sane concurrency range.

**With W-corners (now generated, `configs/k_stress/` has 32 configs, 4 server groups):**
AWQ frees ~10.4GB of weight memory (16.06 -> 5.66 GB), raising the FP16-KV ceiling to
~26 and the FP8-KV ceiling to ~52 -- still inside the same {8,16,32,48} grid. This tests
whether W has its own capacity-relief channel independent of K's precision, not just its
already-measured compute/bandwidth channel.

**Budget:** base grid ~1.5-2h (~20 units, worst-case). With W-corners: ~3-4h total
(~40 units), 4 server launches instead of 2. Independent of Sessions A-C -- run it on
either account, any time.

---
**Re-run note (2026-07-10):** the first attempt returned 16/16 `status: partial` with a
uniform ~9-11% request error rate -- vLLM `400 Bad Request` on documents whose real token
count overshot `max_model_len - max_new_tokens` (word-count sizing underestimated NQ
tokenization). Root cause fixed: documents are now sized **tokenizer-exactly** and every
prompt is verified against a hard `prompt_token_budget: 7900` at build time (a violation
fails the build loudly instead of 400ing mid-run). Re-running this notebook re-executes
all cells (partials are not skipped by resume). Expect `prompt_tokens_mean` ~ 7,460-7,500
in the records -- comfortably under 7,936.


---
**Session scope (2026-07-10):** this notebook now runs THREE corner sets in one session
(40 cells, 6 server launches, ~35–45 units):
1. **K-isolation** (fp16kv/fp8kv × conc {8,16,32,48} × 2 reps) — the capacity experiment.
2. **W capacity corners** (AWQ × both KV dtypes, same grid) — does freeing ~10.4 GB of
   weight memory raise the sustainable-concurrency ceiling (~16 → ~26 predicted)?
3. **KS long-context probe** (EAGLE-3 × both KV dtypes × conc {1, 8} × 2 reps) — the
   K-under-S contrast at 7.4k-token context, same hardware/kernels as the factorial's
   short-context KS (~×0.63 at c1). Deliberately below the capacity ceiling so the
   economics comparison stays clean. Together with the conc-8 K-isolation cells (the
   long-context K-solo measurement, free) this completes the 2×2 grid that decomposes
   emulation tax vs bandwidth credit without an H100.

---
**Bug A/B hardening (2026-07-11):** delete any `configs/k_stress/_parked/` directory from
the previous session — the probe configs are back in the main set and the sweep now
orders them last. Harness fixes: teardown signals the whole process group (vLLM V1's
EngineCore child can no longer be orphaned holding ~16GB), launch refuses to start on an
occupied GPU (one precise error instead of a cascade), a stalled launch fails in ~10 min
instead of 40, and each record now captures the selected attention backend from the
server log (`env.attention_backend`). If the eagle3-fp16kv launch stalls again, see the
"Rung 2" comment in its config files.

In [1]:
# 1. Verify the 40GB card (High-RAM OFF)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
mem = int(subprocess.run(["nvidia-smi","--query-gpu=memory.total","--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().splitlines()[0])
assert mem < 50000, ("Got the 80GB A100 (%d MiB). Toggle High-RAM OFF and reconnect: "
                     "on 80GB the FP16 ceiling (~47) escapes this notebook grid.") % mem
units_before = input("Compute-units balance right now: ")

NVIDIA A100-SXM4-40GB, 40960 MiB
Compute-units balance right now: 130.05


In [15]:
# 2. Repo + Spec-Bench (long documents are built from its RAG passages)
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!git clone --depth 1 https://github.com/hemingkx/Spec-Bench.git external/Spec-Bench 2>/dev/null || echo "Spec-Bench already present"
!test -f external/Spec-Bench/data/spec_bench/question.jsonl && echo "RAG data OK"

/content
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 69 (delta 61), reused 69 (delta 61), pack-reused 0 (from 0)
Unpacking objects: 100% (69/69), 100.42 KiB | 2.51 MiB/s, done.
From https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang
   2c18615..b93f366  main       -> origin/main
Updating 2c18615..b93f366
Fast-forward
 .DS_Store                                          |  Bin 8196 -> 8196 bytes
 HANDOVER.md                                        |   21 +-
 PREREQ_RESULTS.md                                  |   51 +
 colab/phase3b_kstress_40gb.ipynb                   | 1741 +++++++++++---
 configs/k_stress/generate_k_stress.py              |   30 +-
 configs/k_stress/kstress_eagle3-fp16kv_c1_r0.yaml  |    4 +
 configs/k_stress/kstress_eagle3-fp16kv_c1_r1.yaml  |    4 +
 configs/k_stress/kstress_eagle3-fp16kv_c8_r0.yaml  |    4 +
 configs/k_stress/kstress_eagle3-fp16kv_c8_

In [3]:
# 3. Isolated vLLM environment (PREREQ_RESULTS Check 6 recipe; ~6-8 min)
%pip install -q virtualenv
!virtualenv -q /content/vllm_env
!/content/vllm_env/bin/pip install -q vllm==0.24.0 datasets pyyaml requests pytest ninja
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 45.2 MB/s eta 0:00:00
vllm 0.24.0


In [5]:
# 4. HF auth
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

token set: True


In [ ]:
# 4b. Pre-download every checkpoint ONCE, before any server launch --
# bounded, retried, loud. scripts/predownload.py wraps `hf download` with a
# hard per-attempt timeout (30 min), 3 attempts with backoff, and -- new
# since the 2026-07-14 Xet incident -- an AUTOMATIC curl fallback: the
# hf-client route dies with "403 SignatureError: invalid key pair id"
# because the hub sends hf clients to the us.gcp.cdn.hf.co edge (broken
# signing key), while browser-UA requests to plain resolve URLs are 302'd
# to the healthy cas-bridge edge. When the hf CLI exhausts its attempts,
# the script rebuilds the standard HF cache layout (blobs + snapshot
# symlinks + refs) from those resolve URLs, size-verified per file, so the
# sweep proceeds regardless of the broken edge. If hf is KNOWN-broken,
# skip its ~10 min of doomed retries per repo:
#   !/content/vllm_env/bin/python scripts/predownload.py --curl-only
# Re-runs are no-ops on a warm cache. Debug transcript:
# colab/archive_phase3b_xet_debug_20260714.ipynb
!/content/vllm_env/bin/python scripts/predownload.py

[predownload] meta-llama/Llama-3.1-8B-Instruct (timeout 1800s): /content/vllm_env/bin/hf download meta-llama/Llama-3.1-8B-Instruct

Error: (Request ID: 01KXGW9ZEWMJ3ECXJHG5YMZF6D)

403 Forbidden: None.
Cannot access content at: https://us.gcp.cdn.hf.co/xet-bridge-us/6698d8a0653e4babe21e1e7d/b85fa10bd12d4ea01487c18eae95c3261e97514648772d0b2883b67cb164c9d2?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model-00003-of-00004.safetensors%3B+filename%3D%22model-00003-of-00004.safetensors%22%3B&user_id=66f0c636460a6d829091b53c&X-Xet-Cas-Uid=66f0c636460a6d829091b53c&Expires=1784055327&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjY5OGQ4YTA2NTNlNGJhYmUyMWUxZTdkL2I4NWZhMTBiZDEyZDRlYTAxNDg3YzE4ZWFlOTVjMzI2MWU5NzUxNDY0ODc3MmQwYjI4ODNiNjdjYjE2NGM5ZDJcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD02NmYwYzYzNjQ2MGE2ZDgyOTA5MWI1M2MmWC1YZXQtQ2FzLVVpZD02NmYwYzYzNjQ2MGE2ZDgyOTA5MWI1M2MiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
# Check size before committing to the copy
subprocess.run(["du", "-sh", "/root/.cache/huggingface"])

backup_dir = "/content/drive/MyDrive/specdecoding_hf_cache"
!mkdir -p {backup_dir}
!rsync -a --info=progress2 /root/.cache/huggingface/ {backup_dir}/
print("Backup done. Verify:")
!du -sh {backup_dir}

Mounted at /content/drive
 38,735,048,324  99%  191.17MB/s    0:03:13 (xfr#83, to-chk=0/146)
Backup done. Verify:
37G	/content/drive/MyDrive/specdecoding_hf_cache


In [6]:
#4b. Alt: Drive download:
from google.colab import drive
drive.mount('/content/drive')

backup_dir = "/content/drive/MyDrive/specdecoding_hf_cache"
!mkdir -p /root/.cache/huggingface
!rsync -a --info=progress2 {backup_dir}/ /root/.cache/huggingface/
print("Restored. Verify predownload sees everything cached:")
!/content/vllm_env/bin/python scripts/predownload.py

Mounted at /content/drive
 22,674,430,732  99%   73.87MB/s    0:04:52 (xfr#82, to-chk=0/145)
Restored. Verify predownload sees everything cached:
[predownload] meta-llama/Llama-3.1-8B-Instruct (timeout 1800s): /content/vllm_env/bin/hf download meta-llama/Llama-3.1-8B-Instruct

✓ Downloaded
  path: /root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659


[predownload] hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4 (timeout 1800s): /content/vllm_env/bin/hf download hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4

✓ Downloaded
  path: /root/.cache/huggingface/hub/models--hugging-quants--Meta-Llama-3.1-8B-Instruct-AWQ-INT4/snapshots/db1f81ad4b8c7e39777509fac66c652eb0a52f91


[predownload] yuhuili/EAGLE3-LLaMA3.1-Instruct-8B (timeout 1800s): /content/vllm_env/bin/hf download yuhuili/EAGLE3-LLaMA3.1-Instruct-8B

✓ Downloaded
  path: /root/.cache/huggingface/hub/models--yuhuili--EAGLE3-LLaMA3.1-Instruct-8B/snapshots/ada

In [16]:
# 5. Harness self-check (~1 min, no GPU)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m pytest tests -q

........................................................................ [ 46%]
........................................................................ [ 92%]
...........                                                              [100%]
155 passed in 16.11s


In [17]:
# 6. Sanity: the two server commands (identical except --kv-cache-dtype fp8)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/k_stress/kstress_*.yaml" --results-dir results --dry-run 2>&1 | grep "server command"

[sweep] server command: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching --speculative-config {"draft_tensor_parallel_size": 1, "method": "eagle3", "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B", "num_speculative_tokens": 5} --max-num-batched-tokens 8192
[sweep] server command: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching --kv-cache-dtype fp8 --speculative-config {"draft_tensor_parallel_size": 1, "method": "eagle3", "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B", "num_speculative_tokens": 5} --max-num-batched-tokens 8192


In [18]:
# 7. THE SWEEP: 40 cells, 6 launches, resumable. Cells at conc 32/48 under
# FP16-KV are SLOW BY DESIGN (preemption thrash) -- that's the phenomenon,
# not a hang; watch results/server_logs/ if unsure.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/k_stress/kstress_fp16kv_*.yaml" "configs/k_stress/kstress_fp8kv_*.yaml" "configs/k_stress/kstress_w4a16-*.yaml" "configs/k_stress/kstress_eagle3-*.yaml" --results-dir results

[sweep] 40 run(s) in 6 server group(s)
[sweep] skip (complete): k_stress_llama-3-1-8b-instruct_fp16-fp16-none_rag_shared_prefix_c16_r0
[sweep] skip (complete): k_stress_llama-3-1-8b-instruct_fp16-fp16-none_rag_shared_prefix_c16_r1
[sweep] skip (complete): k_stress_llama-3-1-8b-instruct_fp16-fp16-none_rag_shared_prefix_c32_r0
[sweep] skip (complete): k_stress_llama-3-1-8b-instruct_fp16-fp16-none_rag_shared_prefix_c32_r1
[sweep] skip (complete): k_stress_llama-3-1-8b-instruct_fp16-fp16-none_rag_shared_prefix_c48_r0
[sweep] skip (complete): k_stress_llama-3-1-8b-instruct_fp16-fp16-none_rag_shared_prefix_c48_r1
[sweep] skip (complete): k_stress_llama-3-1-8b-instruct_fp16-fp16-none_rag_shared_prefix_c8_r0
[sweep] skip (complete): k_stress_llama-3-1-8b-instruct_fp16-fp16-none_rag_shared_prefix_c8_r1
[sweep] skip (complete): k_stress_llama-3-1-8b-instruct_fp16-fp8-none_rag_shared_prefix_c16_r0
[sweep] skip (complete): k_stress_llama-3-1-8b-instruct_fp16-fp8-none_rag_shared_prefix_c16_r1
[swee

In [19]:
!grep -iE "error|traceback|assert|CUDA" results/server_logs/server_20260715_004237.log | tail -40
!grep -iE "error|traceback|assert|CUDA" results/server_logs/server_20260715_004509.log | tail -40

(APIServer pid=48493) ERROR 07-15 00:44:48 [serving.py:472]   File "/content/vllm_env/lib/python3.12/site-packages/vllm/utils/async_utils.py", line 106, in merge_async_iterators
(APIServer pid=48493) ERROR 07-15 00:44:48 [serving.py:472]     async for item in iterators[0]:
(APIServer pid=48493) ERROR 07-15 00:44:48 [serving.py:472]   File "/content/vllm_env/lib/python3.12/site-packages/vllm/v1/engine/async_llm.py", line 579, in generate
(APIServer pid=48493) ERROR 07-15 00:44:48 [serving.py:472]     out = q.get_nowait() or await q.get()
(APIServer pid=48493) ERROR 07-15 00:44:48 [serving.py:472]                             ^^^^^^^^^^^^^
(APIServer pid=48493) ERROR 07-15 00:44:48 [serving.py:472]   File "/content/vllm_env/lib/python3.12/site-packages/vllm/v1/engine/output_processor.py", line 85, in get
(APIServer pid=48493) ERROR 07-15 00:44:48 [serving.py:472]     raise output
(APIServer pid=48493) ERROR 07-15 00:44:48 [serving.py:472]   File "/content/vllm_env/lib/python3.12/site-pack

In [20]:
!PATH="/content/vllm_env/bin:$PATH" CUDA_LAUNCH_BLOCKING=1 /content/vllm_env/bin/python -m harness.run configs/k_stress/kstress_eagle3-fp16kv_c1_r0.yaml --results-dir /content/debug_results2

[run] launching server: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching --speculative-config {"draft_tensor_parallel_size": 1, "method": "eagle3", "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B", "num_speculative_tokens": 5} --max-num-batched-tokens 8192
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[run] k_stress_llama-3-1-8b-instruct_fp16-fp16-eagle3_rag_shared_prefix_c1_r0: 24 prompts, concurrency=1, max_new_tokens=256
[run] WARNING: spec decoding on but no spec_decode counters found at /m

In [21]:
!ls /content/debug_results2/server_logs/
!grep -iE "error|traceback|assert|CUDA|Triton" /content/debug_results2/server_logs/*.log | tail -60

server_20260715_004950.log
(EngineCore pid=51323) ERROR 07-15 00:51:16 [core.py:1233]     return self.worker.execute_model(scheduler_output)
(EngineCore pid=51323) ERROR 07-15 00:51:16 [core.py:1233]            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=51323) ERROR 07-15 00:51:16 [core.py:1233]   File "/content/vllm_env/lib/python3.12/site-packages/torch/utils/_contextlib.py", line 124, in decorate_context
(EngineCore pid=51323) ERROR 07-15 00:51:16 [core.py:1233]     return func(*args, **kwargs)
(EngineCore pid=51323) ERROR 07-15 00:51:16 [core.py:1233]            ^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=51323) ERROR 07-15 00:51:16 [core.py:1233]   File "/content/vllm_env/lib/python3.12/site-packages/vllm/v1/worker/gpu_worker.py", line 896, in execute_model
(EngineCore pid=51323) ERROR 07-15 00:51:16 [core.py:1233]     output = self.model_runner.execute_model(
(EngineCore pid=51323) ERROR 07-15 00:51:16 [core.py:1233]              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(Engin

In [22]:
import pathlib
src = pathlib.Path("configs/k_stress/kstress_eagle3-fp16kv_c1_r0.yaml")
text = src.read_text().replace(
    'extra: ["--max-num-batched-tokens", "8192"]',
    'extra: ["--max-num-batched-tokens", "8192", "--enforce-eager"]'
)
dst = pathlib.Path("/content/debug_eager_c1_r0.yaml")
dst.write_text(text)
print(dst.read_text())

# K-stress addendum cell, generated by configs/k_stress/generate_k_stress.py.
# Edit the generator, not this file. A100-40GB (Colab High-RAM OFF) -- the
# ceiling arithmetic in the generator docstring assumes the 40GB pool.
block: k_stress
engine: vllm
model: meta-llama/Llama-3.1-8B-Instruct
factors:
  weight_quant: fp16
  kv_quant: fp16
  spec_decode: eagle3
draft_model: yuhuili/EAGLE3-LLaMA3.1-Instruct-8B
workload: rag_shared_prefix
workload_params:
  num_requests: 24
  max_new_tokens: 256
  prefix_overlap: low
  doc_target_tokens: 7400
  # Tokenizer-exact doc sizing + hard prompt budget (Phase-3b root-cause
  # fix): word-approximated sizing let ~10% of docs overshoot
  # max_model_len - max_new_tokens (7936) and draw 400s from vLLM.
  # Budget 7900 = 7936 minus BOS + round-trip jitter margin.
  tokenizer_model: meta-llama/Llama-3.1-8B-Instruct
  prompt_token_budget: 7900
  request_timeout_s: 1800
  spec_bench_file: external/Spec-Bench/data/spec_bench/question.jsonl
concurrency: 1
d

In [23]:
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.run /content/debug_eager_c1_r0.yaml --results-dir /content/debug_results3

[run] launching server: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching --speculative-config {"draft_tensor_parallel_size": 1, "method": "eagle3", "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B", "num_speculative_tokens": 5} --max-num-batched-tokens 8192 --enforce-eager
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[run] k_stress_llama-3-1-8b-instruct_fp16-fp16-eagle3_rag_shared_prefix_c1_r0: 24 prompts, concurrency=1, max_new_tokens=256
[run] k_stress_llama-3-1-8b-instruct_fp16-fp16-eagle3_r

In [11]:
# 8. Pool size from the FP16-KV server log -> predicted-vs-measured plateau
!grep -h "GPU KV cache size" results/server_logs/*.log | tail -2
POOL = input("FP16-KV pool size in tokens (from the FP16 group line above): ").replace(",", "")
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.k_stress results --pool-tokens $POOL

(EngineCore pid=32323) INFO 07-14 23:45:31 [kv_cache_utils.py:2146] GPU KV cache size: 130,912 tokens
(EngineCore pid=33370) INFO 07-14 23:47:57 [kv_cache_utils.py:2146] GPU KV cache size: 261,824 tokens


KeyboardInterrupt: Interrupted by user

In [12]:
# 9. Preserve everything
units_after = input("Compute-units balance now: ")
print("Units burned:", units_before, "->", units_after)
!zip -qr phase3b_kstress_results.zip results
from google.colab import files
files.download("phase3b_kstress_results.zip")

Compute-units balance now: 120
Units burned: 130.05 -> 120


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reading the result

The four-row table should tell a two-knee story:

- **conc 8:** ratio ≈ 0.95–1.05, kv-usage well below 1.0, zero preemptions on both —
  the Phase-2 null reproducing below the ceiling. This row is what makes the rest
  credible.
- **conc 16:** FP16 kv-usage max ≈ 1.0 for the first time (batch pinned at the pool
  limit), FP8 cruising at ~0.5. Goodput ratio starts moving.
- **conc 32:** maximum divergence — FP16 batch flat at ~16 with preemptions > 0 and TTFT
  p95 blowing up; FP8 batch ~32, hitting ITS knee (kv-usage → 1.0).
- **conc 48:** both plateaued, FP8 plateau ≈ 2× FP16 — "FP8-KV doubles sustainable
  concurrency" measured on both sides, not extrapolated.

Cross-check the predicted plateau line (pool ÷ measured mean context) against the
measured batch columns; agreement within ~10% closes the loop between the capacity
arithmetic and the observed behavior. Decision-guide sentence this feeds: *enable FP8-KV
when projected demand (concurrency × context × 128 KiB) approaches the pool; below that
it costs ~5% on A100 (emulation tax) and buys nothing.*

**W capacity section:** the AWQ table's FP16-KV column should plateau near ~26 (vs ~16
for FP16 weights) — weight quantization as a KV-capacity lever, measured.

**KS probe section:** the decisive column is the K-under-S ratio vs the K-solo ratio at
the same concurrency, both at long context. If K-under-S climbs from the factorial's
~×0.63 toward the K-solo value, context length buys back bandwidth credit and QuantSpec's
regime-dependence is confirmed; if it stays near ×0.63 while τ stays flat, the emulation
tax dominates at any context length on A100. Either outcome is a clean sentence for the
decision guide.